# Docling section-preserving DOCX POC

This notebook trims an inclusive, 1-based page range from a PDF, converts only those pages with Docling, creates an editable section-preserving DOCX, and chunks that DOCX by its Word heading hierarchy. It also saves the selected-page PDF, a Markdown layout preview, section index, chunk JSON, and an Excel workbook with `chunk_id` and `chunk_text`.

Set `PDF_PATH`, `START_PAGE`, and `END_PAGE` in the setup cell. Each PDF and selected page range gets its own folder under `outputs/section_docx_poc`.


In [2]:
from __future__ import annotations

from collections import Counter
from io import BytesIO
from pathlib import Path
import json
import sys

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docx import Document
from docx.shared import Inches, Pt
from pypdf import PdfReader, PdfWriter

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
PDF_PATH = Path("/Users/suyashutekar/Projects/pdflens_replacement/rbi_docs/32.pdf")
START_PAGE = 4  # 1-based, inclusive
END_PAGE = 10    # 1-based, inclusive; set to None to use the final PDF page

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}. Set PDF_PATH to your PDF.")

source_reader = PdfReader(str(PDF_PATH))
SOURCE_PAGE_COUNT = len(source_reader.pages)
if END_PAGE is None:
    END_PAGE = SOURCE_PAGE_COUNT
if not (1 <= START_PAGE <= END_PAGE <= SOURCE_PAGE_COUNT):
    raise ValueError(
        f"Invalid page range {START_PAGE}-{END_PAGE}; PDF has {SOURCE_PAGE_COUNT} pages. "
        "Page numbers are 1-based and inclusive."
    )

PAGE_RANGE_LABEL = f"pages_{START_PAGE}-{END_PAGE}"
OUTPUT_DIR = (
    PROJECT_ROOT / "outputs" / "section_docx_poc"
    / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRIMMED_PDF_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}.pdf"
DOCX_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_sections.docx"
SECTION_INDEX_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_sections.json"
MARKDOWN_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_{PAGE_RANGE_LABEL}_layout.md"
CHUNK_JSON_PATH = OUTPUT_DIR / f"{DOCX_PATH.stem}_chunks.json"
CHUNK_XLSX_PATH = OUTPUT_DIR / f"{DOCX_PATH.stem}_chunks.xlsx"

writer = PdfWriter()
for source_index in range(START_PAGE - 1, END_PAGE):
    writer.add_page(source_reader.pages[source_index])
with TRIMMED_PDF_PATH.open("wb") as trimmed_pdf_file:
    writer.write(trimmed_pdf_file)

print(f"Input PDF: {PDF_PATH}")
print(f"Selected source pages: {START_PAGE}-{END_PAGE} of {SOURCE_PAGE_COUNT}")
print(f"Trimmed PDF: {TRIMMED_PDF_PATH}")
print(f"DOCX: {DOCX_PATH}")


Input PDF: /Users/suyashutekar/Projects/pdflens_replacement/rbi_docs/32.pdf
Selected source pages: 4-10 of 56
Trimmed PDF: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10.pdf
DOCX: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10_sections.docx


## 1. Convert the selected pages with Docling

`START_PAGE` and `END_PAGE` are 1-based and inclusive. The selected pages are first copied into a separate PDF; Docling, furniture extraction, DOCX generation, and chunking then use that trimmed PDF/DOCX.


In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.generate_parsed_pages = True
pipeline_options.heading_hierarchy_options.enabled = True
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)
conversion = converter.convert(str(TRIMMED_PDF_PATH))
doc = conversion.document

print(f"Selected pages processed: {doc.num_pages()} (source pages {START_PAGE}-{END_PAGE})")
print(f"Text items: {len(doc.texts)} | Tables: {len(doc.tables)} | Pictures: {len(doc.pictures)}")


W1026 16:03:22.088000 13199 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 13014.24it/s]
/Users/suyashutekar/Projects/pdflens_replacement/.venv/lib/python3.14/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1105.)
  return F.conv2d(


Selected pages processed: 7 (source pages 4-10)
Text items: 141 | Tables: 0 | Pictures: 7


## 2. Inspect detected sections

Review Docling's heading labels, levels, and page provenance before generating the DOCX.


In [4]:
def label_name(item):
    label = getattr(item, "label", "")
    return str(getattr(label, "name", getattr(label, "value", label))).upper()

def first_page_number(item):
    provenance = getattr(item, "prov", None) or []
    return getattr(provenance[0], "page_no", None) if provenance else None

heading_preview = []
for item, tree_level in doc.iterate_items():
    if label_name(item) in {"SECTION_HEADER", "TITLE"}:
        heading_preview.append({
            "page": (first_page_number(item) + START_PAGE - 1)
                    if first_page_number(item) is not None else None,
            "level": getattr(item, "level", tree_level or 1),
            "label": label_name(item),
            "text": getattr(item, "text", ""),
            "top": float(getattr((getattr(item, "prov", None) or [None])[0], "bbox", None).t)
                   if (getattr(item, "prov", None) or []) else 0,
        })

heading_preview.sort(key=lambda heading: (heading["page"] or 0, -heading["top"]))
print(f"Detected {len(heading_preview)} title/section headings")
for heading in heading_preview[:80]:
    level = max(1, min(int(heading["level"] or 1), 9))
    print(f"source p.{heading['page']} | {'  ' * (level - 1)}{heading['text']}")


Detected 4 title/section headings
source p.4 | Chapter I - Preliminary
source p.4 |   A. Short Title and Commencement
source p.4 |   B. Applicability
source p.6 |   C. Definitions


## 2. Build the section-preserving DOCX for the selected pages

Docling content and PDF text-layer furniture are grouped by trimmed-page order, then written into separate Word sections. The section index records original source-page numbers using the selected-range offset.


In [5]:
from collections import defaultdict
import re
import unicodedata

import pdfplumber
from docx.enum.section import WD_SECTION
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt, RGBColor
from docx.oxml.ns import qn

word = Document()
word.core_properties.title = PDF_PATH.stem
word.core_properties.subject = "Docling section-preservation proof of concept"

first_page = next(iter(doc.pages.values()), None)
if first_page is not None and getattr(first_page, "size", None) is not None:
    section = word.sections[0]
    section.page_width = Pt(float(first_page.size.width))
    section.page_height = Pt(float(first_page.size.height))
    section.top_margin = Inches(0.55)
    section.bottom_margin = Inches(0.55)
    section.left_margin = Inches(0.58)
    section.right_margin = Inches(0.58)

# Compact, readable typography helps each source page fit on its corresponding DOCX page.
normal = word.styles["Normal"]
normal.font.name = "Arial"
normal.font.size = Pt(9)
normal.paragraph_format.space_after = Pt(3)
normal.paragraph_format.line_spacing = 1.0
title_style = word.styles["Title"]
title_style.font.color.rgb = RGBColor(0, 0, 0)
title_style.font.underline = False
title_ppr = title_style._element.get_or_add_pPr()
title_border = title_ppr.find(qn("w:pBdr"))
if title_border is not None:
    title_ppr.remove(title_border)

for style_name, size in (("Title", 18), ("Heading 1", 14), ("Heading 2", 12),
                         ("Heading 3", 10.5), ("Heading 4", 9.5), ("Heading 5", 9),
                         ("Heading 6", 9), ("Heading 7", 9), ("Heading 8", 9),
                         ("Heading 9", 9)):
    style = word.styles[style_name]
    style.font.name = "Arial"
    style.font.size = Pt(size)
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(2)
    style.paragraph_format.keep_with_next = True


def normalize(value):
    value = unicodedata.normalize("NFKC", str(value or "")).casefold()
    value = re.sub(r"\(?cid:\d+\)?", " ", value)
    value = re.sub(r"^\s*\d+[.)]\s+", " ", value)
    return re.sub(r"[^\w]+", " ", value).strip()


def clean_pdf_text(value):
    return re.sub(r"\(?cid:\d+\)?", "•", str(value or "")).strip()


def pdf_page_lines(page):
    """Group PDF words into visual lines while retaining top-origin coordinates."""
    words = sorted(page.extract_words(), key=lambda item: (item["top"], item["x0"]))
    lines = []
    for word_item in words:
        match = next((line for line in reversed(lines)
                      if abs(line["top"] - word_item["top"]) <= 2.5), None)
        if match is None:
            match = {"top": word_item["top"], "words": []}
            lines.append(match)
        match["words"].append(word_item)
    for line in lines:
        line["words"].sort(key=lambda item: item["x0"])
        line["text"] = " ".join(item["text"] for item in line["words"])
    return sorted(lines, key=lambda line: line["top"])


pdf_page_data = []
with pdfplumber.open(TRIMMED_PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        page_lines = pdf_page_lines(page)
        top_band = page.height * 0.055
        bottom_band = page.height * 0.055
        header_lines = [line for line in page_lines if line["top"] <= top_band]
        footer_lines = [line for line in page_lines
                        if line["top"] >= page.height - bottom_band]
        pdf_page_data.append({
            "width": page.width,
            "height": page.height,
            "header": "   |   ".join(clean_pdf_text(line["text"]) for line in header_lines),
            "footer": "   |   ".join(clean_pdf_text(line["text"]) for line in footer_lines),
            "furniture_text": " ".join(
                line["text"] for line in header_lines + footer_lines
            ),
            "body_lines": [line for line in page_lines
                           if line not in header_lines and line not in footer_lines],
        })

markdown_parts = [
    f"# {PDF_PATH.stem} layout and section preview",
    f"Source PDF: `{PDF_PATH.name}`  \nSelected source pages: {START_PAGE}-{END_PAGE}",
    "## Page headers and footers",
]
for selected_page, page_data in enumerate(pdf_page_data, start=1):
    source_page = selected_page + START_PAGE - 1
    markdown_parts.append(f"### Source page {source_page}")
    if page_data["header"]:
        markdown_parts.append(f"**Header:** {page_data['header']}")
    if page_data["footer"]:
        markdown_parts.append(f"**Footer:** {page_data['footer']}")
markdown_parts.extend(["## Docling content", doc.export_to_markdown()])
MARKDOWN_PATH.write_text("\n\n".join(markdown_parts).rstrip() + "\n", encoding="utf-8")
print(f"Saved Markdown layout preview: {MARKDOWN_PATH}")

items_by_page = defaultdict(list)
recognized_text_by_page = defaultdict(list)
for item, tree_level in doc.iterate_items():
    page_no = first_page_number(item)
    if page_no is None:
        continue
    provenance = (getattr(item, "prov", None) or [None])[0]
    bbox = getattr(provenance, "bbox", None) if provenance else None
    # Docling coordinates are bottom-origin; negative top sorts topmost first.
    top = float(getattr(bbox, "t", 0)) if bbox is not None else 0.0
    left = float(getattr(bbox, "l", 0)) if bbox is not None else 0.0
    items_by_page[page_no].append((top, left, item, tree_level))
    text = str(getattr(item, "text", "") or "").strip()
    if text:
        recognized_text_by_page[page_no].append(text)
    if label_name(item) in {"TABLE", "TABLE_ITEM"}:
        recognized_text_by_page[page_no].append(item.export_to_markdown(doc=doc))

# Reconcile the PDF's selectable text with Docling's output. Add any complete
# source text line that Docling did not emit, preserving its page and position.
for page_number, page_data in enumerate(pdf_page_data, start=1):
    recognized_text = normalize(" ".join(recognized_text_by_page[page_number]))
    structured_regions = []
    for item, _tree_level in doc.iterate_items():
        if label_name(item) not in {"TABLE", "TABLE_ITEM", "PICTURE", "PICTURE_ITEM"}:
            continue
        provenance = (getattr(item, "prov", None) or [None])[0]
        if provenance is None or provenance.page_no != page_number:
            continue
        bbox = getattr(provenance, "bbox", None)
        if bbox is not None:
            structured_regions.append(bbox)
    for line in page_data["body_lines"]:
        line_text = clean_pdf_text(line["text"])
        normalized_line = normalize(line_text)
        if not normalized_line or normalized_line in recognized_text:
            continue
        line_left = min(word["x0"] for word in line["words"])
        line_right = max(word["x1"] for word in line["words"])
        line_top = page_data["height"] - line["top"]
        line_bottom = page_data["height"] - max(word["bottom"] for word in line["words"])
        already_in_structured_region = any(
            line_right >= float(bbox.l) - 4 and line_left <= float(bbox.r) + 4
            and line_top >= float(bbox.b) - 28 and line_bottom <= float(bbox.t) + 28
            for bbox in structured_regions
        )
        if not already_in_structured_region:
            items_by_page[page_number].append(
                (line_top, line_left,
                 {"label": "PDF_FALLBACK", "text": line_text}, 1)
            )

section_index = []
block_counts = Counter()


def set_page_furniture(section, page_data):
    section.header.is_linked_to_previous = False
    section.footer.is_linked_to_previous = False
    header = section.header.paragraphs[0]
    header.alignment = WD_ALIGN_PARAGRAPH.CENTER
    header.paragraph_format.space_after = Pt(2)
    run = header.add_run(page_data["header"])
    run.font.name = "Arial"
    run.font.size = Pt(7)
    footer = section.footer.paragraphs[0]
    footer.alignment = WD_ALIGN_PARAGRAPH.CENTER
    footer.paragraph_format.space_before = Pt(2)
    run = footer.add_run(page_data["footer"])
    run.font.name = "Arial"
    run.font.size = Pt(7)


def record_block(category):
    block_counts[category] += 1


def add_native_table(item):
    cells = list(getattr(getattr(item, "data", None), "table_cells", None) or [])
    if not cells:
        fallback = item.export_to_markdown(doc=doc)
        if fallback.strip():
            word.add_paragraph(fallback)
        return bool(fallback.strip())
    rows = max(int(cell.start_row_offset_idx) + max(1, int(cell.row_span or 1)) for cell in cells)
    cols = max(int(cell.start_col_offset_idx) + max(1, int(cell.col_span or 1)) for cell in cells)
    table = word.add_table(rows=rows, cols=cols)
    table.style = "Table Grid"
    for cell in cells:
        row = int(cell.start_row_offset_idx)
        col = int(cell.start_col_offset_idx)
        row_span = max(1, int(cell.row_span or 1))
        col_span = max(1, int(cell.col_span or 1))
        target = table.cell(row, col)
        if row_span > 1 or col_span > 1:
            target = target.merge(table.cell(row + row_span - 1, col + col_span - 1))
        target.text = str(cell.text or "")
        for paragraph in target.paragraphs:
            paragraph.paragraph_format.space_after = Pt(0)
            for run in paragraph.runs:
                run.font.name = "Arial"
                run.font.size = Pt(8)
                if bool(cell.column_header or cell.row_header):
                    run.bold = True
    return True


def add_picture(item):
    image = item.get_image(doc)
    if image is None:
        return False
    stream = BytesIO()
    image.save(stream, format="PNG")
    stream.seek(0)
    provenance = (getattr(item, "prov", None) or [None])[0]
    bbox = getattr(provenance, "bbox", None) if provenance else None
    page_width = pdf_page_data[first_page_number(item) - 1]["width"]
    detected_width = float(getattr(bbox, "width", 0)) / 72 if bbox else 0
    max_width = (page_width - 2 * 0.58 * 72) / 72
    picture_width = min(max(detected_width, 0.1), max_width)
    word.add_picture(stream, width=Inches(picture_width))
    return True


for page_number in range(1, doc.num_pages() + 1):
    if page_number > 1:
        word.add_section(WD_SECTION.NEW_PAGE)
    page_section = word.sections[-1]
    set_page_furniture(page_section, pdf_page_data[page_number - 1])

    page_items = sorted(items_by_page.get(page_number, []), key=lambda entry: (-entry[0], entry[1]))
    for _top, _left, item, tree_level in page_items:
        label = ("PDF_FALLBACK" if isinstance(item, dict) and item.get("label") == "PDF_FALLBACK"
                 else label_name(item))
        text = str(item.get("text", "") if isinstance(item, dict)
                   else getattr(item, "text", "") or "").strip()
        if (text and label not in {"SECTION_HEADER", "TITLE"}
                and normalize(text) in normalize(pdf_page_data[page_number - 1]["furniture_text"])):
            continue

        if label in {"SECTION_HEADER", "TITLE"} and text:
            # Place the document's detected title at the start of its page instead of
            # relying on Docling's tree order, which can yield it after body content.
            level = max(1, min(int(getattr(item, "level", tree_level or 1) or 1), 9))
            if page_number == 1 and level == 1 and not section_index:
                word.add_paragraph(text, style="Title")
                stored_level = 1
            else:
                word.add_heading(text, level=min(level, 9))
                stored_level = level
            source_page_number = page_number + START_PAGE - 1
            section_index.append({"title": text, "level": stored_level,
                                  "start_page": source_page_number,
                                  "end_page": source_page_number})
            record_block("heading")
            continue

        if label in {"TABLE", "TABLE_ITEM"}:
            record_block("structured_table" if add_native_table(item) else "table_markdown_fallback")
            continue
        if label in {"PICTURE", "PICTURE_ITEM"}:
            if add_picture(item):
                record_block("picture")
            continue
        if not text:
            continue
        if label in {"LIST_ITEM", "ORDERED_LIST", "UNORDERED_LIST"}:
            style = "List Number" if label == "ORDERED_LIST" else "List Bullet"
            paragraph = word.add_paragraph(text, style=style)
            record_block("list_item")
        elif label in {"FOOTNOTE", "FOOTNOTE_REFERENCE"}:
            paragraph = word.add_paragraph()
            run = paragraph.add_run(text)
            run.italic = True
            run.font.size = Pt(8)
            record_block("footnote")
        else:
            word.add_paragraph(text)
            record_block("pdf_text_fallback" if label == "PDF_FALLBACK" else "text")

word.save(DOCX_PATH)
SECTION_INDEX_PATH.write_text(json.dumps(section_index, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved: {DOCX_PATH}")
print(f"Saved: {SECTION_INDEX_PATH}")
print(f"Content blocks: {dict(block_counts)}")
print(f"Section headings: {len(section_index)}")


Saved: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10_sections.docx
Saved: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10_sections.json
Content blocks: {'picture': 7, 'text': 68, 'heading': 4, 'list_item': 38, 'pdf_text_fallback': 30}
Section headings: 4


## 3. Chunk the selected-range DOCX by section

The chunker reads the generated DOCX, prefixes each chunk with its heading path using ` | ` separators, and saves both JSON and an Excel workbook.


In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from chunking_strategy.sectionwise_chunking import chunk_docx, save_chunks_xlsx

check = Document(DOCX_PATH)
heading_counts = Counter(
    paragraph.style.name for paragraph in check.paragraphs
    if paragraph.style.name.startswith("Heading ")
)
print(f"DOCX size: {DOCX_PATH.stat().st_size:,} bytes")
print(f"Word heading styles: {dict(heading_counts)}")
print(f"Native Word tables: {len(check.tables)}")
print(f"Indexed sections: {len(json.loads(SECTION_INDEX_PATH.read_text(encoding='utf-8')))}")

if not section_index:
    print("WARNING: Docling detected no section headings. The output cannot preserve a hierarchy it did not identify.")
elif not check.paragraphs:
    raise RuntimeError("Generated DOCX has no paragraphs.")

chunk_result = chunk_docx(DOCX_PATH)
CHUNK_JSON_PATH.write_text(
    json.dumps(chunk_result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
save_chunks_xlsx(chunk_result["chunks"], CHUNK_XLSX_PATH)
print(f"Section chunks: {chunk_result['chunk_count']}")
print(f"Chunk JSON: {CHUNK_JSON_PATH}")
print(f"Chunk Excel: {CHUNK_XLSX_PATH}")


DOCX size: 53,822 bytes
Word heading styles: {'Heading 2': 3}
Native Word tables: 0
Indexed sections: 4
Section chunks: 5
Chunk JSON: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10_sections_chunks.json
Chunk Excel: /Users/suyashutekar/Projects/pdflens_replacement/outputs/section_docx_poc/32_pages_4-10/32_pages_4-10_sections_chunks.xlsx


## Notes

- Page ranges are 1-based and inclusive. Set `END_PAGE = None` to include every page from `START_PAGE` through the end of the PDF.
- The trimmed PDF and a Markdown layout preview are saved as separate outputs; the original PDF is not modified. The Markdown shows Docling headings/tables and the recovered per-page headers/footers.
- The section index and DOCX headings use source PDF page numbers. The chunker groups content using the Word heading hierarchy and writes the heading path plus body text separated by ` | `.
- DOCX pagination remains editable/reflowable within the selected page range. Dense source pages may flow differently in Word.
